In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# --- Load Data ---
df = pd.read_excel('../data/Online Retail.xlsx', engine='openpyxl')

# --- First look ---
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nFirst 5 rows:")
df.head()

Shape: (541909, 8)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data Types:
 InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

First 5 rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# --- Missing values ---
print("=== MISSING VALUES ===")
print(df.isnull().sum())
print(f"\nMissing % of CustomerID: {df['CustomerID'].isnull().mean()*100:.1f}%")

# --- Basic stats ---
print("\n=== BASIC STATS ===")
print(f"Date range: {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
print(f"Unique countries: {df['Country'].nunique()}")
print(f"Unique products: {df['StockCode'].nunique()}")
print(f"Total transactions: {df['InvoiceNo'].nunique()}")

=== MISSING VALUES ===
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Missing % of CustomerID: 24.9%

=== BASIC STATS ===
Date range: 2010-12-01 08:26:00 → 2011-12-09 12:50:00
Unique customers: 4372
Unique countries: 38
Unique products: 4070
Total transactions: 25900


In [3]:
# --- Step 2: Data Cleaning ---

print(f"Original shape: {df.shape}")

# 1. Drop rows with no CustomerID (guest checkouts — can't track for RFM)
df = df.dropna(subset=['CustomerID'])
print(f"After dropping missing CustomerID: {df.shape}")

# 2. Drop cancellations (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After dropping cancellations: {df.shape}")

# 3. Drop rows where Quantity <= 0 or UnitPrice <= 0
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
print(f"After dropping bad Quantity/UnitPrice: {df.shape}")

# 4. Convert CustomerID to integer
df['CustomerID'] = df['CustomerID'].astype(int)

# 5. Create TotalPrice column (needed for Monetary)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f"\nFinal clean shape: {df.shape}")
print(f"Unique customers remaining: {df['CustomerID'].nunique()}")
print(f"\nSample TotalPrice values:")
print(df[['InvoiceNo','CustomerID','Quantity','UnitPrice','TotalPrice']].head())


Original shape: (541909, 8)
After dropping missing CustomerID: (406829, 8)
After dropping cancellations: (397924, 8)
After dropping bad Quantity/UnitPrice: (397884, 8)

Final clean shape: (397884, 9)
Unique customers remaining: 4338

Sample TotalPrice values:
  InvoiceNo  CustomerID  Quantity  UnitPrice  TotalPrice
0    536365       17850         6       2.55       15.30
1    536365       17850         6       3.39       20.34
2    536365       17850         8       2.75       22.00
3    536365       17850         6       3.39       20.34
4    536365       17850         6       3.39       20.34


In [4]:
# --- Step 3: RFM Calculation ---

import datetime as dt

# Reference date: 1 day after the last transaction in the dataset
reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
print(f"Reference date: {reference_date}")

# Build RFM table
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)

print(f"\nRFM Table Shape: {rfm.shape}")
print(f"\nRFM Stats:")
print(rfm[['Recency','Frequency','Monetary']].describe().round(2))
print(f"\nFirst 5 rows:")
print(rfm.head())

Reference date: 2011-12-10 12:50:00

RFM Table Shape: (4338, 4)

RFM Stats:
       Recency  Frequency   Monetary
count  4338.00    4338.00    4338.00
mean     92.54       4.27    2054.27
std     100.01       7.70    8989.23
min       1.00       1.00       3.75
25%      18.00       1.00     307.41
50%      51.00       2.00     674.48
75%     142.00       5.00    1661.74
max     374.00     209.00  280206.02

First 5 rows:
   CustomerID  Recency  Frequency  Monetary
0       12346      326          1  77183.60
1       12347        2          7   4310.00
2       12348       75          4   1797.24
3       12349       19          1   1757.55
4       12350      310          1    334.40


In [5]:
# --- Step 4: RFM Scoring ---

# Recency: LOWER is better (bought recently = good), so reverse the scoring
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=5, labels=[5,4,3,2,1])

# Frequency: HIGHER is better
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])

# Monetary: HIGHER is better
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  q=5, labels=[1,2,3,4,5])

# Convert to int
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)

# Combined RFM Score (string) and total score (numeric)
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Score']   = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print("Score distribution:")
print(rfm[['R_Score','F_Score','M_Score','RFM_Score']].describe().round(2))
print(f"\nSample rows:")
print(rfm.head(10))

Score distribution:
       R_Score  F_Score  M_Score  RFM_Score
count  4338.00  4338.00  4338.00    4338.00
mean      3.02     3.00     3.00       9.02
std       1.41     1.41     1.41       3.59
min       1.00     1.00     1.00       3.00
25%       2.00     2.00     2.00       6.00
50%       3.00     3.00     3.00       9.00
75%       4.00     4.00     4.00      12.00
max       5.00     5.00     5.00      15.00

Sample rows:
   CustomerID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0       12346      326          1  77183.60        1        1        5   
1       12347        2          7   4310.00        5        5        5   
2       12348       75          4   1797.24        2        4        4   
3       12349       19          1   1757.55        4        1        4   
4       12350      310          1    334.40        1        1        2   
5       12352       36          8   2506.04        3        5        5   
6       12353      204          1     89.00        1

In [6]:
# --- Step 5: Customer Segmentation ---

def segment_customer(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']
    score = row['RFM_Score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent Customers'
    elif r >= 3 and f <= 2 and m <= 2:
        return 'Promising'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'Hibernating'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Segment summary
segment_summary = rfm.groupby('Segment').agg(
    Customer_Count = ('CustomerID', 'count'),
    Avg_Recency    = ('Recency',    'mean'),
    Avg_Frequency  = ('Frequency',  'mean'),
    Avg_Monetary   = ('Monetary',   'mean'),
    Total_Revenue  = ('Monetary',   'sum')
).round(2).reset_index()

segment_summary['Revenue_%'] = (segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum() * 100).round(1)
segment_summary = segment_summary.sort_values('Total_Revenue', ascending=False)

print(segment_summary.to_string(index=False))

         Segment  Customer_Count  Avg_Recency  Avg_Frequency  Avg_Monetary  Total_Revenue  Revenue_%
       Champions             962        12.86          11.08       6038.82     5809341.07       65.2
 Loyal Customers             758        35.69           4.13       1842.60     1396691.42       15.7
         At Risk             454       141.63           3.81       1634.69      742149.95        8.3
            Lost            1356       173.14           1.51        313.21      424717.31        4.8
     Hibernating             241       181.63           1.30       1367.79      329637.70        3.7
Recent Customers             319        18.52           1.24        458.20      146166.57        1.6
       Promising             248        53.37           1.07        252.84       62703.88        0.7


In [7]:
# --- Step 6: Visualizations ---

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Plot 1: Customer Count by Segment ---
fig1 = px.bar(
    segment_summary.sort_values('Customer_Count', ascending=True),
    x='Customer_Count', y='Segment', orientation='h',
    title='Customer Count by Segment',
    color='Customer_Count',
    color_continuous_scale='Blues',
    text='Customer_Count'
)
fig1.update_traces(textposition='outside')
fig1.update_layout(showlegend=False, height=400)
fig1.show()

# --- Plot 2: Revenue % by Segment ---
fig2 = px.pie(
    segment_summary,
    values='Revenue_%', names='Segment',
    title='Revenue Contribution by Segment (%)',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig2.show()

# --- Plot 3: RFM Score Distribution ---
fig3 = px.histogram(
    rfm, x='RFM_Score', nbins=13,
    title='Distribution of RFM Scores',
    color_discrete_sequence=['#2196F3']
)
fig3.update_layout(bargap=0.1)
fig3.show()

# --- Plot 4: Recency vs Monetary (colored by segment) ---
fig4 = px.scatter(
    rfm, x='Recency', y='Monetary',
    color='Segment', size='Frequency',
    title='Recency vs Monetary Value (size = Frequency)',
    hover_data=['CustomerID'],
    opacity=0.6
)
fig4.update_layout(height=500)
fig4.show()

print("All 4 charts rendered.")

All 4 charts rendered.


In [9]:
# --- Step 7: Business Recommendations ---

recommendations = {
    'Champions': {
        'count': 962,
        'revenue_pct': 65.2,
        'action': 'Reward & Retain',
        'recommendation': (
            "Champions generate 65.2% of all revenue despite being only 22% of customers. "
            "Launch a VIP loyalty program with early access to new products, exclusive discounts, "
            "and personalised thank-you communications. Losing even 10% of Champions = ~£580K revenue loss."
        )
    },
    'Loyal Customers': {
        'count': 758,
        'revenue_pct': 15.7,
        'action': 'Upsell & Elevate',
        'recommendation': (
            "758 Loyal Customers contribute 15.7% of revenue with avg order frequency of 4.13. "
            "Target with bundle offers and cross-sell campaigns to push them into Champion tier. "
            "A 20% conversion to Champions adds ~£800K in annual revenue."
        )
    },
    'At Risk': {
        'count': 454,
        'revenue_pct': 8.3,
        'action': 'Win-Back Campaign',
        'recommendation': (
            "454 At Risk customers averaged £1,634 spend but haven't purchased in 141 days. "
            "Deploy targeted win-back emails with time-limited discount (10-15%). "
            "Even 30% reactivation recovers ~£222K in at-risk revenue."
        )
    },
    'Hibernating': {
        'count': 241,
        'revenue_pct': 3.7,
        'action': 'Re-engagement',
        'recommendation': (
            "241 Hibernating customers spent avg £1,367 but are 181 days inactive. "
            "Send a 'We miss you' campaign with a strong incentive. "
            "Higher spend history suggests these were once valuable — worth reactivation attempt before writing off."
        )
    },
    'Lost': {
        'count': 1356,
        'revenue_pct': 4.8,
        'action': 'Suppress or Survey',
        'recommendation': (
            "1,356 Lost customers (largest group) contribute only 4.8% of revenue. "
            "Do not invest heavily in reactivation. Send a single exit survey to understand churn reasons. "
            "Redirect saved marketing budget to Champions retention and At Risk win-back."
        )
    },
    'Recent Customers': {
        'count': 319,
        'revenue_pct': 1.6,
        'action': 'Nurture & Convert',
        'recommendation': (
            "319 Recent Customers just entered the funnel with low frequency (1.24 avg orders). "
            "Trigger onboarding email sequence within 7 days of first purchase. "
            "Goal: convert to Promising or Loyal tier within 60 days."
        )
    },
    'Promising': {
        'count': 248,
        'revenue_pct': 0.7,
        'action': 'Activate',
        'recommendation': (
            "248 Promising customers are newest and lowest spend. "
            "Offer first-repeat-purchase incentive (free shipping or small discount). "
            "Early activation here builds the future Champion pipeline."
        )
    }
}

print("=" * 70)
print("BUSINESS RECOMMENDATIONS — RFM SEGMENTATION")
print("=" * 70)
for segment, data in recommendations.items():
    print(f"\n[{data['action'].upper()}] {segment} ({data['count']} customers | {data['revenue_pct']}% revenue)")
    print(f"  → {data['recommendation']}")

print("\n" + "=" * 70)
print(f"TOTAL CUSTOMERS ANALYSED: 4,338")
print(f"TOTAL REVENUE MAPPED: £8,910,510.48")
print(f"KEY INSIGHT: Top 39.7% of customers (Champions + Loyal) drive 80.9% of revenue")
print("=" * 70)

BUSINESS RECOMMENDATIONS — RFM SEGMENTATION

[REWARD & RETAIN] Champions (962 customers | 65.2% revenue)
  → Champions generate 65.2% of all revenue despite being only 22% of customers. Launch a VIP loyalty program with early access to new products, exclusive discounts, and personalised thank-you communications. Losing even 10% of Champions = ~£580K revenue loss.

[UPSELL & ELEVATE] Loyal Customers (758 customers | 15.7% revenue)
  → 758 Loyal Customers contribute 15.7% of revenue with avg order frequency of 4.13. Target with bundle offers and cross-sell campaigns to push them into Champion tier. A 20% conversion to Champions adds ~£800K in annual revenue.

[WIN-BACK CAMPAIGN] At Risk (454 customers | 8.3% revenue)
  → 454 At Risk customers averaged £1,634 spend but haven't purchased in 141 days. Deploy targeted win-back emails with time-limited discount (10-15%). Even 30% reactivation recovers ~£222K in at-risk revenue.

[RE-ENGAGEMENT] Hibernating (241 customers | 3.7% revenue)
  → 2

In [10]:
# Save clean data and RFM table
rfm.to_csv('../data/rfm_scored.csv', index=False)
segment_summary.to_csv('../data/segment_summary.csv', index=False)
print("Files saved.")

Files saved.
